# Chapter 15: Deep Learning on Unseen Data — Introducing Federated Learning
<a href="https://colab.research.google.com/github/SatrioArjunaPutra/GrokkingDeepLearning/blob/main/Chapter%2015%20-%20Deep%20Learning%20on%20Unseen%20Data/15_federated_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Course:** Tugas 2 - Enrichment Deep Learning Class  
**Student Name:** Satrio Arjuna Putra  
**Student ID (NIM):** 101032330178  
**Class:** TK-47-05  
**Primary Reference:** *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)

---

## Chapter Objectives
In this final chapter of *Grokking Deep Learning*, we address one of the most critical challenges in modern artificial intelligence: **privacy-preserving deep learning** on decentralized, unseen data.

1. **The Privacy Dilemma:** Why centralized data collection poses severe privacy, regulatory, and security risks in real-world applications.
2. **Federated Learning (FL) Paradigm:** Shifting computation to the edge — bringing the model to the data instead of the data to the model.
3. **Federated Averaging (FedAvg):** Implementing the foundational consensus algorithm for aggregating decentralized model weights in pure NumPy.
4. **Non-IID Data Dynamics:** Understanding how heterogeneous data distributions across edge clients affect convergence.
5. **Secure Multi-Party Aggregation (SecAgg):** Cryptographic intuition of pairwise zero-sum noise masks protecting weight updates.
6. **Differential Privacy (DP-FedAvg):** Calibrated Gaussian noise addition and gradient clipping to provide rigorous mathematical privacy guarantees.
7. **Comparative Benchmarking:** Evaluating Centralized vs. Standard FedAvg vs. DP-FedAvg with empirical convergence curves.

## 1. Theoretical Foundations & Mathematical Formulations

### 1.1 The Privacy Problem in Centralized Machine Learning
Traditional deep learning operates under a **centralized assumption**: all training data must be gathered into a single data lake or central server:

$$\mathcal{D}_{\text{central}} = \bigcup_{k=1}^K \mathcal{D}_k$$

While computationally convenient, this paradigm presents critical vulnerabilities:
- **Regulatory Constraints:** Compliance frameworks such as GDPR, HIPAA, and CCPA strictly prohibit the transmission of identifiable user data (e.g., patient health records, financial transactions, keyboard keystrokes).
- **Single Point of Failure:** Central databases represent high-value targets for data breaches.
- **Bandwidth Constraints:** Transmitting gigabytes of raw sensor or video data from edge devices (smartphones, IoT) to the cloud is bandwidth-prohibitive.

---

### 1.2 The Federated Learning Paradigm
Federated Learning (McMahan et al., 2017) fundamentally reverses the data flow:
> *Instead of bringing raw data to the model, we bring the model to the data.*

```
                      [Central Server]
                     Global Model: W_t
                         /   |   \
             Download   /    |    \  Download
             Weights   /     |     \ Weights
                      v      v      v
                  Client 1 Client 2 Client 3
                  (Data 1) (Data 2) (Data 3)
                      |      |      |
                  Local SGD Local SGD Local SGD
                      |      |      |
             Upload   \      |      / Upload
             Updates   \     |     /  Updates
                        v    v    v
                      [Central Server]
                 W_{t+1} = FedAvg(W_t^1, W_t^2, ...)
```

---

### 1.3 The Federated Averaging (FedAvg) Algorithm
Suppose there are $K$ decentralized clients, where client $k$ holds a private local dataset $\mathcal{D}_k$ containing $n_k = |\mathcal{D}_k|$ samples. The total number of samples is $n = \sum_{k=1}^K n_k$.

The global objective is to minimize the aggregate loss across all clients:
$$\min_{\mathbf{w}} f(\mathbf{w}) = \sum_{k=1}^K \frac{n_k}{n} F_k(\mathbf{w}), \quad \text{where } F_k(\mathbf{w}) = \frac{1}{n_k} \sum_{i \in \mathcal{D}_k} \ell(\mathbf{w}; x_i, y_i)$$

In each communication round $t = 0, 1, \dots, T-1$:
1. **Server Broadcast:** The central server distributes current global weights $\mathbf{w}_t$ to all participating clients.
2. **Local Client Optimization:** Each client $k$ initializes local weights $\mathbf{w}_{t, 0}^k = \mathbf{w}_t$ and performs $E$ epochs of local mini-batch SGD:
   $$\mathbf{w}_{t, e+1}^k \leftarrow \mathbf{w}_{t, e}^k - \eta \nabla F_k(\mathbf{w}_{t, e}^k)$$
3. **Weight Upload:** Each client sends its updated local model $\mathbf{w}_{t+1}^k$ (or weight delta $\Delta \mathbf{w}_k = \mathbf{w}_{t+1}^k - \mathbf{w}_t$) back to the server.
4. **Server Aggregation (FedAvg):** The server computes a weighted average of the client models:
   $$\mathbf{w}_{t+1} = \sum_{k=1}^K \frac{n_k}{n} \mathbf{w}_{t+1}^k$$

---

### 1.4 The Non-IID Data Challenge
In real-world decentralized environments, data across clients is **Non-IID** (Not Independent and Identically Distributed):
- **Label Distribution Skew:** Hospital A specializes in cardiology (high rate of Class 0), while Hospital B specializes in oncology (high rate of Class 1).
- **Feature Distribution Skew:** Speech recognizers on mobile phones handle drastically different user accents, ambient noise levels, and vocabulary.
- When data is Non-IID, local gradients point in divergent directions ($
abla F_j(\mathbf{w}) \ne \nabla F_k(\mathbf{w})$). Aggregating divergent weights introduces client drift, requiring careful tuning of local epochs $E$ and learning rate $\eta$.

---

### 1.5 Secure Aggregation (SecAgg): Zero-Sum Pairwise Masking
Merely sharing weight updates is **not fully private**: curious servers can perform *gradient inversion attacks* (e.g., Deep Leakage from Gradients) to reconstruct private training samples.

**Secure Multi-Party Aggregation (SecAgg)** solves this via pairwise zero-sum masks. If Client $i$ and Client $j$ share a negotiated secret random seed $r_{ij} \sim \mathcal{N}(0, \sigma^2)$ such that $r_{ji} = -r_{ij}$, Client $i$ transmits a masked weight:
$$\tilde{\mathbf{w}}_i = \mathbf{w}_i + \sum_{j \ne i} r_{ij}$$

When the central server sums all masked submissions:
$$\sum_{i=1}^K \tilde{\mathbf{w}}_i = \sum_{i=1}^K \mathbf{w}_i + \sum_{i=1}^K \sum_{j \ne i} r_{ij} = \sum_{i=1}^K \mathbf{w}_i + 0 = \sum_{i=1}^K \mathbf{w}_i$$

The random masks cancel out perfectly to zero! The server recovers the exact sum $\sum \mathbf{w}_i$ without ever observing any individual client's private model $\mathbf{w}_i$.

---

### 1.6 Differential Privacy (DP-FedAvg)
Even when individual weights are hidden, the global model itself might memorize private training examples. **Differential Privacy (DP)** provides a rigorous mathematical bound:

> An algorithm $\mathcal{M}$ satisfies $(\epsilon, \delta)$-Differential Privacy if for any two adjacent datasets $D, D'$ differing by a single sample, and for any output set $S \subseteq \text{Range}(\mathcal{M})$:
> $$\mathbb{P}[\mathcal{M}(D) \in S] \le e^{\epsilon} \mathbb{P}[\mathcal{M}(D') \in S] + \delta$$

In **DP-FedAvg**, privacy is enforced through two mandatory steps before transmission:
1. **$L_2$ Norm Clipping:** Ensure the sensitivity of local updates is bounded by threshold $S$:
   $$\bar{\mathbf{w}}_k = \frac{\mathbf{w}_k}{\max\left(1, \frac{\|\mathbf{w}_k\|_2}{S}\right)}$$
2. **Calibrated Gaussian Perturbation:** Inject zero-mean Gaussian noise scaled by the privacy noise level $\sigma_{dp}$:
   $$\mathbf{w}_{k}^{\text{priv}} = \bar{\mathbf{w}}_k + \mathcal{N}\left(0, \sigma_{dp}^2 S^2 \mathbf{I}\right)$$
This trades a small, controlled amount of model accuracy for formal, provable privacy guarantees.

In [ ]:
# Chapter 15: Environment Setup & Imports
import numpy as np
import matplotlib.pyplot as plt

# Set fixed random seed for 100% reproducible scientific experiments
np.random.seed(42)
print("NumPy successfully imported. Version:", np.__version__)
print("Differential Privacy & Federated Learning Simulation Initialized.")

In [ ]:
# Step 1: Synthetic Decentralized Non-IID Multi-Client Dataset Generation
# We simulate 5 decentralized clients (e.g., 5 regional hospitals or private users)
# Each client has a private local distribution with significant Non-IID label skew.

def generate_client_data(num_clients=5, samples_per_client=200, input_dim=8):
    """
    Generates non-IID datasets partitioned across decentralized edge clients.
    Each client has a different bias toward positive (Class 1) or negative (Class 0) samples,
    simulating real-world heterogeneous hospital distributions.
    """
    clients_X = []
    clients_y = []
    
    # Fixed underlying feature transformation weight
    true_w = np.random.randn(input_dim, 1) * 0.8
    
    # Class 1 ratios across clients (Non-IID skew from 10% to 90%)
    class_1_ratios = [0.10, 0.30, 0.50, 0.70, 0.90]
    
    for k in range(num_clients):
        p_pos = class_1_ratios[k]
        n_pos = int(samples_per_client * p_pos)
        n_neg = samples_per_client - n_pos
        
        # Positive samples: shifted toward +true_w
        X_pos = np.random.randn(n_pos, input_dim) * 0.9 + 0.6 * true_w.T
        y_pos = np.ones((n_pos, 1))
        
        # Negative samples: shifted away from true_w
        X_neg = np.random.randn(n_neg, input_dim) * 0.9 - 0.6 * true_w.T
        y_neg = np.zeros((n_neg, 1))
        
        # Concatenate and shuffle
        X = np.vstack([X_pos, X_neg])
        y = np.vstack([y_pos, y_neg])
        indices = np.random.permutation(samples_per_client)
        
        clients_X.append(X[indices])
        clients_y.append(y[indices])
        
    # Also create a held-out balanced test set for global model validation
    n_test = 500
    X_test_pos = np.random.randn(n_test // 2, input_dim) * 0.9 + 0.6 * true_w.T
    X_test_neg = np.random.randn(n_test // 2, input_dim) * 0.9 - 0.6 * true_w.T
    X_test = np.vstack([X_test_pos, X_test_neg])
    y_test = np.vstack([np.ones((n_test // 2, 1)), np.zeros((n_test // 2, 1))])
    test_idx = np.random.permutation(n_test)
    X_test = X_test[test_idx]
    y_test = y_test[test_idx]
    
    return clients_X, clients_y, X_test, y_test, class_1_ratios

clients_X, clients_y, X_test, y_test, class_ratios = generate_client_data()
print("Synthetic Multi-Client Dataset Generated Successfully:")
for i in range(len(clients_X)):
    pos_ratio = np.mean(clients_y[i]) * 100
    print(f"  - Client {i+1}: {clients_X[i].shape[0]} samples | Positives: {pos_ratio:.1f}% | Non-IID Ratio: {class_ratios[i]:.2f}")
print(f"Global Balanced Test Set: {X_test.shape[0]} samples, Positives: {np.mean(y_test)*100:.1f}%")

In [ ]:
# Step 2: Scratch Multilayer Perceptron (MLP) Model Architecture
# Built entirely with NumPy matrices, implementing sigmoid forward pass,
# binary cross-entropy loss, and analytical backpropagation.

class MLPModel:
    def __init__(self, input_dim=8, hidden_dim=16, output_dim=1):
        # Weight initialization (He / Xavier standard scaling)
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        
    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -15.0, 15.0)))
    
    @staticmethod
    def relu(z):
        return np.maximum(0, z)
    
    @staticmethod
    def relu_deriv(z):
        return (z > 0).astype(float)
    
    def get_weights(self):
        """Extract flat copy of model parameters."""
        return {
            'W1': self.W1.copy(),
            'b1': self.b1.copy(),
            'W2': self.W2.copy(),
            'b2': self.b2.copy()
        }
    
    def set_weights(self, weights_dict):
        """Assign parameter copies into current model."""
        self.W1 = weights_dict['W1'].copy()
        self.b1 = weights_dict['b1'].copy()
        self.W2 = weights_dict['W2'].copy()
        self.b2 = weights_dict['b2'].copy()
        
    def forward(self, X):
        """Forward propagation through two layers."""
        # Shape: (N, input_dim) @ (input_dim, hidden_dim) -> (N, hidden_dim)
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.relu(self.z1)
        
        # Shape: (N, hidden_dim) @ (hidden_dim, output_dim) -> (N, output_dim)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2
    
    def backward(self, X, y, lr=0.05):
        """Backpropagation and parameter update via Stochastic Gradient Descent."""
        N = X.shape[0]
        
        # Output delta: dL/dz2 for Binary Cross-Entropy with Sigmoid
        # Shape: (N, 1)
        dz2 = (self.a2 - y) / N
        
        # Gradients for Layer 2
        dW2 = np.dot(self.a1.T, dz2)  # Shape: (hidden_dim, 1)
        db2 = np.sum(dz2, axis=0, keepdims=True)  # Shape: (1, 1)
        
        # Hidden delta via chain rule
        da1 = np.dot(dz2, self.W2.T)  # Shape: (N, hidden_dim)
        dz1 = da1 * self.relu_deriv(self.z1)  # Shape: (N, hidden_dim)
        
        # Gradients for Layer 1
        dW1 = np.dot(X.T, dz1)  # Shape: (input_dim, hidden_dim)
        db1 = np.sum(dz1, axis=0, keepdims=True)  # Shape: (1, hidden_dim)
        
        # Gradient descent update
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        
    def evaluate(self, X, y):
        """Computes binary cross entropy loss and classification accuracy."""
        preds = self.forward(X)
        eps = 1e-12
        loss = -np.mean(y * np.log(preds + eps) + (1.0 - y) * np.log(1.0 - preds + eps))
        acc = np.mean((preds >= 0.5).astype(float) == y)
        return loss, acc

print("MLP Model Architecture constructed successfully.")

In [ ]:
# Step 3: Centralized Learning Baseline (Ideal Privacy-Unfriendly Upper Bound)
# In this baseline, all clients pool their raw private data into one central database.

def train_centralized(clients_X, clients_y, X_test, y_test, epochs=30, lr=0.08):
    # Pool all client data together
    X_pool = np.vstack(clients_X)
    y_pool = np.vstack(clients_y)
    
    model = MLPModel(input_dim=X_pool.shape[1], hidden_dim=16, output_dim=1)
    
    history_loss = []
    history_acc = []
    
    N = X_pool.shape[0]
    batch_size = 32
    
    for epoch in range(epochs):
        idx = np.random.permutation(N)
        for b_start in range(0, N, batch_size):
            b_idx = idx[b_start:b_start + batch_size]
            xb, yb = X_pool[b_idx], y_pool[b_idx]
            model.forward(xb)
            model.backward(xb, yb, lr=lr)
            
        test_loss, test_acc = model.evaluate(X_test, y_test)
        history_loss.append(test_loss)
        history_acc.append(test_acc)
        
    return model, history_loss, history_acc

print("Training Centralized Baseline Model...")
central_model, central_loss, central_acc = train_centralized(clients_X, clients_y, X_test, y_test, epochs=30)
print(f"Centralized Final Test Accuracy: {central_acc[-1]*100:.2f}%")
print(f"Centralized Final Test Loss: {central_loss[-1]:.4f}")

In [ ]:
# Step 4: Federated Averaging (FedAvg) Algorithm Simulation
# Clients train strictly on local data; only model weights are transmitted to the server.

class FederatedClient:
    def __init__(self, client_id, X, y):
        self.client_id = client_id
        self.X = X
        self.y = y
        self.num_samples = X.shape[0]
        self.model = MLPModel(input_dim=X.shape[1], hidden_dim=16, output_dim=1)
        
    def local_train(self, global_weights, local_epochs=3, lr=0.08, batch_size=32):
        """
        Initialize local model with global parameters, run local SGD for E epochs,
        and return updated local weights.
        """
        self.model.set_weights(global_weights)
        
        for _ in range(local_epochs):
            indices = np.random.permutation(self.num_samples)
            for start in range(0, self.num_samples, batch_size):
                b_idx = indices[start:start + batch_size]
                xb, yb = self.X[b_idx], self.y[b_idx]
                self.model.forward(xb)
                self.model.backward(xb, yb, lr=lr)
                
        return self.model.get_weights()


class FederatedServer:
    def __init__(self, input_dim=8, hidden_dim=16, output_dim=1):
        self.global_model = MLPModel(input_dim, hidden_dim, output_dim)
        
    def get_global_weights(self):
        return self.global_model.get_weights()
    
    def federated_average(self, client_weights_list, client_sample_sizes):
        """
        Implements the FedAvg mathematical formulation:
        W_{t+1} = sum_{k=1}^K (n_k / n) * W_k
        """
        total_samples = sum(client_sample_sizes)
        new_weights = {}
        
        # Initialize aggregated parameter tensors to zero
        for key in client_weights_list[0].keys():
            new_weights[key] = np.zeros_like(client_weights_list[0][key])
            
        # Weighted accumulation
        for client_w, n_k in zip(client_weights_list, client_sample_sizes):
            weight_factor = n_k / total_samples
            for key in new_weights.keys():
                new_weights[key] += weight_factor * client_w[key]
                
        self.global_model.set_weights(new_weights)
        return new_weights
    
    def evaluate(self, X_test, y_test):
        return self.global_model.evaluate(X_test, y_test)

# Run Standard FedAvg Experiment
def run_fedavg_simulation(clients_X, clients_y, X_test, y_test, rounds=30, local_epochs=3, lr=0.08):
    server = FederatedServer(input_dim=X_test.shape[1], hidden_dim=16, output_dim=1)
    clients = [FederatedClient(i, clients_X[i], clients_y[i]) for i in range(len(clients_X))]
    client_sizes = [c.num_samples for c in clients]
    
    fed_acc_history = []
    fed_loss_history = []
    
    for r in range(rounds):
        global_w = server.get_global_weights()
        
        # Step 1 & 2: Clients train locally
        client_updates = []
        for client in clients:
            local_w = client.local_train(global_w, local_epochs=local_epochs, lr=lr)
            client_updates.append(local_w)
            
        # Step 3: Server aggregates via FedAvg
        server.federated_average(client_updates, client_sizes)
        
        # Evaluate global model on unseen test set
        test_loss, test_acc = server.evaluate(X_test, y_test)
        fed_loss_history.append(test_loss)
        fed_acc_history.append(test_acc)
        
    return server, fed_loss_history, fed_acc_history

print("Running Standard Federated Averaging (FedAvg) Simulation...")
fed_server, fed_loss, fed_acc = run_fedavg_simulation(clients_X, clients_y, X_test, y_test, rounds=30)
print(f"FedAvg Final Global Test Accuracy: {fed_acc[-1]*100:.2f}%")
print(f"FedAvg Final Global Test Loss: {fed_loss[-1]:.4f}")

In [ ]:
# Step 5: Differentially Private Federated Learning (DP-FedAvg)
# To prevent memorization and membership inference attacks, each client:
# 1. Clips local weight updates by an L2 norm bound S.
# 2. Injects calibrated Gaussian noise scaled by sigma_dp.

def apply_differential_privacy(local_weights, global_weights, clip_norm=1.5, noise_scale=0.08):
    """
    Enforces Differential Privacy on weight delta: delta_W = W_local - W_global.
    Step 1: Calculate L2 norm of the update vector.
    Step 2: Clip delta_W if norm > clip_norm.
    Step 3: Add calibrated Gaussian noise ~ N(0, (noise_scale * clip_norm)^2).
    """
    dp_weights = {}
    
    # Compute total L2 norm across all parameter layers
    total_norm_sq = 0.0
    deltas = {}
    for k in local_weights.keys():
        deltas[k] = local_weights[k] - global_weights[k]
        total_norm_sq += np.sum(deltas[k] ** 2)
    total_norm = np.sqrt(total_norm_sq)
    
    # L2 Clipping scaling factor
    clip_coef = min(1.0, clip_norm / (total_norm + 1e-8))
    
    # Additive Gaussian noise standard deviation
    sigma = noise_scale * clip_norm
    
    for k in local_weights.keys():
        # Clipped update
        clipped_delta = deltas[k] * clip_coef
        # Noise injection
        noise = np.random.normal(0.0, sigma, size=clipped_delta.shape)
        dp_weights[k] = global_weights[k] + (clipped_delta + noise)
        
    return dp_weights

def run_dp_fedavg_simulation(clients_X, clients_y, X_test, y_test, noise_scale=0.08, rounds=30, local_epochs=3, lr=0.08):
    server = FederatedServer(input_dim=X_test.shape[1], hidden_dim=16, output_dim=1)
    clients = [FederatedClient(i, clients_X[i], clients_y[i]) for i in range(len(clients_X))]
    client_sizes = [c.num_samples for c in clients]
    
    dp_acc_history = []
    dp_loss_history = []
    
    for r in range(rounds):
        global_w = server.get_global_weights()
        client_updates = []
        
        for client in clients:
            local_w = client.local_train(global_w, local_epochs=local_epochs, lr=lr)
            # Apply DP Mechanism before sending update to central server
            priv_w = apply_differential_privacy(local_w, global_w, clip_norm=1.2, noise_scale=noise_scale)
            client_updates.append(priv_w)
            
        server.federated_average(client_updates, client_sizes)
        
        test_loss, test_acc = server.evaluate(X_test, y_test)
        dp_loss_history.append(test_loss)
        dp_acc_history.append(test_acc)
        
    return dp_loss_history, dp_acc_history

# Test different noise scales representing increasing privacy budgets
print("Running DP-FedAvg Simulations across different noise levels...")
noise_levels = [0.03, 0.08, 0.20]
dp_results = {}
for n_scale in noise_levels:
    print(f"  -> Simulating DP-FedAvg with Noise Scale = {n_scale}...")
    loss_hist, acc_hist = run_dp_fedavg_simulation(clients_X, clients_y, X_test, y_test, noise_scale=n_scale, rounds=30)
    dp_results[n_scale] = {'loss': loss_hist, 'acc': acc_hist}
    print(f"     Final Accuracy at sigma={n_scale}: {acc_hist[-1]*100:.2f}%")

In [ ]:
# Step 6: Secure Multi-Party Aggregation (SecAgg) Zero-Sum Masking Simulation
# We mathematically demonstrate how pairwise secret masks cancel out perfectly upon aggregation.

def simulate_secure_aggregation(num_clients=4, param_dim=5):
    print("--- Secure Aggregation (SecAgg) Demonstration ---")
    # Simulated raw client model vectors
    raw_weights = [np.round(np.random.randn(param_dim) * 2.0, 2) for _ in range(num_clients)]
    true_sum = np.sum(raw_weights, axis=0)
    
    print("1. True Client Weight Vectors (Private):")
    for i, w in enumerate(raw_weights):
        print(f"   Client {i+1} raw weights: {w}")
    print(f"   -> True Unmasked Sum:   {true_sum}\n")
    
    # Generate pairwise negotiated random masks r_{ij} where r_{ji} = -r_{ij}
    masks = np.zeros((num_clients, num_clients, param_dim))
    for i in range(num_clients):
        for j in range(i + 1, num_clients):
            r = np.round(np.random.randn(param_dim) * 5.0, 2)
            masks[i, j] = r
            masks[j, i] = -r
            
    # Each client creates an encrypted/masked transmission vector
    transmitted_vectors = []
    print("2. Transmitted Masked Vectors (Encrypted / Noise-Obfuscated):")
    for i in range(num_clients):
        net_mask = np.sum(masks[i], axis=0)
        masked_vector = raw_weights[i] + net_mask
        transmitted_vectors.append(masked_vector)
        print(f"   Client {i+1} transmits: {masked_vector}  (looks like pure random noise!)")
        
    # Server aggregates by simply summing the transmitted vectors
    server_sum = np.sum(transmitted_vectors, axis=0)
    print("\n3. Server Aggregation Result:")
    print(f"   -> Server Recovered Sum: {server_sum}")
    print(f"   -> Absolute Difference:  {np.max(np.abs(server_sum - true_sum)):.1e}")
    print("\nResult: The pairwise zero-sum masks cancel out exactly: Sum_masks = 0.0!")
    print("The server reconstructs the exact sum without ever observing any client's individual vector.")

simulate_secure_aggregation()

In [ ]:
# Step 7: Analytical Visualizations & Experimental Comparison
plt.style.use('default')
fig = plt.figure(figsize=(16, 12), dpi=120)

# Plot 1: Non-IID Class Imbalance across decentralized clients
ax1 = fig.add_subplot(2, 2, 1)
client_labels = [f"Client {i+1}" for i in range(len(clients_X))]
pos_counts = [np.sum(clients_y[i] == 1) for i in range(len(clients_X))]
neg_counts = [np.sum(clients_y[i] == 0) for i in range(len(clients_X))]
x_idx = np.arange(len(client_labels))
bar_width = 0.35
ax1.bar(x_idx - bar_width/2, pos_counts, bar_width, label='Class 1 (Positive)', color='#2ca02c', alpha=0.85)
ax1.bar(x_idx + bar_width/2, neg_counts, bar_width, label='Class 0 (Negative)', color='#d62728', alpha=0.85)
ax1.set_xticks(x_idx)
ax1.set_xticklabels(client_labels, fontweight='bold')
ax1.set_ylabel('Sample Count', fontsize=11)
ax1.set_title('(A) Non-IID Label Skew Across Decentralized Clients', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

# Plot 2: Convergence Comparison: Centralized vs FedAvg vs DP-FedAvg
ax2 = fig.add_subplot(2, 2, 2)
rounds_range = np.arange(1, len(central_acc) + 1)
ax2.plot(rounds_range, [a * 100 for a in central_acc], 'k--', linewidth=2.5, label='Centralized Baseline (No Privacy)')
ax2.plot(rounds_range, [a * 100 for a in fed_acc], color='#1f77b4', linewidth=2.5, label='Standard FedAvg (Distributed)')
colors = ['#ff7f0e', '#9467bd', '#8c564b']
for (n_scale, res), c in zip(dp_results.items(), colors):
    ax2.plot(rounds_range, [a * 100 for a in res['acc']], color=c, linewidth=1.8, linestyle='-.', 
             label=f'DP-FedAvg ($\sigma={n_scale}$)')
ax2.set_xlabel('Communication Rounds / Epochs', fontsize=11)
ax2.set_ylabel('Global Test Accuracy (%)', fontsize=11)
ax2.set_title('(B) Convergence Dynamics & Accuracy Benchmarking', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=9.5)
ax2.grid(True, linestyle='--', alpha=0.5)

# Plot 3: Privacy-Utility Tradeoff (Noise Scale vs Final Accuracy)
ax3 = fig.add_subplot(2, 2, 3)
all_noises = [0.0] + list(dp_results.keys())
all_accuracies = [fed_acc[-1] * 100] + [dp_results[s]['acc'][-1] * 100 for s in dp_results.keys()]
ax3.plot(all_noises, all_accuracies, marker='o', markersize=8, color='#d62728', linewidth=2)
for x, y in zip(all_noises, all_accuracies):
    ax3.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 10), ha='center', fontweight='bold')
ax3.set_xlabel('DP Noise Scale ($\sigma$) [Stronger Privacy $\rightarrow$]', fontsize=11)
ax3.set_ylabel('Final Test Accuracy (%)', fontsize=11)
ax3.set_title('(C) Fundamental Privacy-Utility Tradeoff', fontsize=12, fontweight='bold')
ax3.grid(True, linestyle='--', alpha=0.5)

# Plot 4: Test Loss Trajectories Comparison
ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(rounds_range, central_loss, 'k--', linewidth=2.0, label='Centralized Loss')
ax4.plot(rounds_range, fed_loss, color='#1f77b4', linewidth=2.0, label='FedAvg Loss')
for (n_scale, res), c in zip(dp_results.items(), colors):
    ax4.plot(rounds_range, res['loss'], color=c, linewidth=1.5, linestyle=':', label=f'DP Loss ($\sigma={n_scale}$)')
ax4.set_xlabel('Communication Rounds / Epochs', fontsize=11)
ax4.set_ylabel('Binary Cross-Entropy Loss', fontsize=11)
ax4.set_title('(D) Loss Minimization Trajectories', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right', fontsize=9.5)
ax4.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 2. Key Takeaways & Critical Evaluation

### Core Architectural Insights

1. **Decentralized Learning is Viable and Robust:**
   - Even when client datasets are highly Non-IID (ranging from 10% to 90% positive label skew), **Federated Averaging (FedAvg)** converges successfully to an accuracy close to the centralized upper bound without a single raw data point ever leaving client storage.

2. **The Defense-in-Depth Privacy Stack:**
   - **Federated Learning alone is not complete privacy:** Shared gradients can still leak training data through gradient inversion attacks.
   - **Secure Aggregation (SecAgg)** hides individual updates from the server using zero-sum pairwise cryptographic masks.
   - **Differential Privacy (DP-FedAvg)** ensures the aggregated global model does not memorize individual private samples by enforcing bounded sensitivity ($L_2$ clipping) and injecting calibrated Gaussian noise.

3. **The Fundamental Privacy-Utility Tradeoff:**
   - As shown in experimental Plot (C), increasing the noise scale $\sigma_{dp}$ provides stronger privacy bounds ($\epsilon \to 0$), but directly degrades model classification accuracy.
   - Production systems select $\epsilon$ carefully based on regulatory risk and required accuracy thresholds.

---

### Real-World Production Challenges
- **Communication Overhead:** Transmitting millions of parameters over mobile cellular connections requires gradient compression, quantization, and sparse communication.
- **Client Heterogeneity & Stragglers:** Edge devices possess vastly different computational capabilities and network battery states. Asynchronous aggregation and client selection algorithms are essential.
- **Byzantine & Poisoning Attacks:** Malicious clients can submit crafted gradients to corrupt the global consensus. Robust aggregation rules (e.g., Krum, Trimmed Mean, Median) provide Byzantine fault tolerance.

---

### Closing Reflections on *Grokking Deep Learning*
From Chapter 1's simplest forward propagation and single-parameter error reduction to Chapter 15's multi-party decentralized federated learning with differential privacy, we have constructed every fundamental block of modern deep learning **strictly from scratch in pure NumPy**:
- **Ch 01–06:** Mathematical foundations (Forward Prop, Gradient Descent, Multi-variable calculus, Backprop, ReLU).
- **Ch 07–10:** Computer vision & architectural primitives (Picture weights, Dropout regularization, Mini-batching, Activation functions, 2D Convolutions & Max Pooling).
- **Ch 11–15:** Advanced NLP, Optimization Engines, & Privacy (Word2Vec CBOW embeddings, Recurrent Neural Networks, Automatic Differentiation Tensor Framework, Long Short-Term Memory, and Federated Learning).

---

*This completes the entire code reproduction and theoretical synthesis of Grokking Deep Learning (Manning Publications) for Tugas 2 Deep Learning Enrichment Class.*